# Debug v3 Endpoint Dataset
最小验证：直接从 v3 bezier 读 endpoint dataset，不走 source edge。

In [ ]:
from pathlib import Path
import sys
import matplotlib.pyplot as plt
import numpy as np

REPO_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from edge_datasets.endpoint_datamodule import EndpointDetectionDataModule


In [ ]:
FIXTURE_ROOT = REPO_ROOT / 'outputs' / 'v3_dataloader_profile_fixture'
dataset_cfg = {
    'dataset_type': 'laion_synthetic',
    'data_root': str(FIXTURE_ROOT),
    'image_root': str(FIXTURE_ROOT / 'edge_detection'),
    'bezier_root': str(FIXTURE_ROOT / 'laion_edge_v3_bezier'),
    'entry_cache_path': str(FIXTURE_ROOT / 'laion_entry_cache_v3_bezier.txt'),
    'batch_glob': 'batch*',
    'selection_seed': 20260318,
    'selection_offset': 0,
    'max_samples': 8,
}
config = {
    'data': {
        'include_primary_train_dataset': False,
        'rgb_input': True,
        'image_size': 256,
        'batch_size': 8,
        'val_batch_size': 8,
        'num_workers': 0,
        'pin_memory': False,
        'persistent_workers': False,
        'target_degree': 5,
        'min_curve_length': 3.0,
        'max_targets': 512,
        'endpoint_dedupe_distance_px': 2.0,
        'train_augment': False,
        'augment': {},
        'extra_train_datasets': [dataset_cfg],
        'val_dataset': dataset_cfg,
        'test_dataset': dataset_cfg,
    },
    'model': {'arch': 'dab_endpoint_detr'},
    'trainer': {},
}
dm = EndpointDetectionDataModule(config)
dm.setup('fit')
batch = next(iter(dm.val_dataloader()))
images, targets = batch['images'], batch['targets']
print(images.shape)
print(sorted(targets[0].keys()))
print(targets[0]['sample_id'], targets[0]['points'].shape, targets[0]['image_size'])


In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.reshape(-1)
for idx, ax in enumerate(axes):
    image = images[idx].permute(1, 2, 0).cpu().numpy()
    pts = targets[idx]['points'].cpu().numpy().copy()
    h, w = image.shape[:2]
    pts[:, 0] *= w
    pts[:, 1] *= h
    ax.imshow(image)
    if pts.size:
        ax.scatter(pts[:, 0], pts[:, 1], s=16, c='r')
    ax.set_title(str(targets[idx]['sample_id']))
    ax.axis('off')
plt.tight_layout()
